# Project: Personal Chef

Capstone for Module 1 — combine a web-search tool, a domain system prompt, and short-term memory into a leftover-ingredient recipe assistant.

**Workflow**
1. Load env + define a Tavily `web_search` tool
2. Write a chef system prompt that requires searching for recipes
3. Create an agent with tools + `InMemorySaver`
4. Ask about leftovers and inspect the tool-calling trace


In [ ]:
from dotenv import load_dotenv

load_dotenv()

### Define the search tool

Reuse the Tavily-backed `web_search` tool so the chef can look up real recipes.


In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

### System prompt

Instruct the agent to search before suggesting recipes, then return suggestions (and full instructions on request).


In [ ]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

### Create the chef agent

Wire model, tools, prompt, and checkpointer together.


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),  # remember ingredients across turns
)


### Run a conversation turn

Use a stable `thread_id` so follow-ups can refine the recipe choice.


In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

### Inspect the full response

Pretty-print the message history to see search calls and the final answer.


In [ ]:
from pprint import pprint

pprint(response)